# V01 -- Results audit (v20.45)
Every saved result checked: an SE of ~0 is NOT a result (fill values, or one cluster); a model SE far below the design-based SE; effects indistinguishable from 0; outcomes whose fill years were removed. -> results\\RESULTS_AUDIT.md

In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")

sys.path.insert(0,"../All_DIDsCodes"); sys.path.insert(0,"All_DIDsCodes")
import numpy as np, pandas as pd
import _common as C
C.require_engine("20.44")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
C.start_cell_log("V01")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
C.autotune()          # v17: measures THIS machine (cores/RAM/VRAM/disk) and sets every performance knob
C.step(1,"TWFE must EQUAL the closed-form 2x2 DiD identity")
rng=np.random.default_rng(99); n=37; data=[]; means={}
for tr in [0,1]:
    for po in [0,1]:
        v=rng.normal(10+2*tr+3*po+5*tr*po,2,n); means[(tr,po)]=v.mean()
        data+= [{"y":x,"treat":tr,"post":po} for x in v]
d=pd.DataFrame(data); d["did_term"]=d.treat*d.post
beta,_=C.estimate_twfe_did(d,"y","did_term","treat","post","treat")
closed=(means[(1,1)]-means[(1,0)])-(means[(0,1)]-means[(0,0)])
diff=abs(beta-closed)
print(f"closed-form = {closed:.12f}\nTWFE        = {beta:.12f}\ndifference  = {diff:.3e}")
(C.ok if diff<1e-9 else C.fail)(f"TWFE == closed-form 2x2 identity (diff {diff:.2e})")
assert diff<1e-9
C.done(1, next_task="next cell")


In [ ]:
import _common as C
C.start_cell_log("V01")
audit = C.audit_results()
print(audit[audit.flags != "ok"].to_string(index=False) if len(audit) else "no results yet")
